# Análise Exploratória de Dados (EDA) Avançada: Indicadores Educacionais por Região
## Diagnóstico Estatístico, Desigualdades Estruturais e Séries Temporais (INEP / MEC)
---
**Papéis:** Cientista de Dados Sênior, Engenheiro de Dados e Estatístico
**Base Primária:** `br_inep_indicadores_educacionais_regiao.csv`

### Objetivo desta Investigação:
Transformar a base histórica de mais de 200 variáveis educacionais do Brasil em diagnósticos visuais e quantitativos precisos. Esta análise investiga:
1. **Estatísticas Descritivas:** Média, mediana, desvio-padrão, percentis e assimetria das taxas de aprovação, abandono e distorção idade-série (TDI).
2. **Detecção de Outliers e Anomalias:** Aplicação do método de Amplitude Interquartil (IQR) para identificar gargalos escolares extremos.
3. **Matriz de Correlação Multivariada:** Avaliação da hipótese pedagógica da correlação entre qualificação docente (DSU), horas-aula diárias (HAD) e rendimento escolar.
4. **Disparidades Regionais e Dependência Administrativa:** O abismo de oportunidades entre as redes pública e privada e zonas urbana vs rural.
5. **Série Temporal (2013-2024):** A recuperação dos fluxos escolares na última década e o impacto pós-pandemia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configurações estéticas do Matplotlib / Seaborn
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 5)

# Ingestão segura dos dados brutos com auditoria de integridade
df_raw = pd.read_csv('br_inep_indicadores_educacionais_regiao.csv', low_memory=False)
print(f"Dimensões brutas do arquivo: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas.")

# Deduplicação e higienização
df = df_raw.drop_duplicates(subset=['ano', 'regiao', 'localizacao', 'rede']).copy()
print(f"Dimensões pós-deduplicação: {df.shape[0]} linhas × {df.shape[1]} colunas.")

### 1. Tratamento de Tipos e Seleção das Métricas Fundamentais

In [ ]:
colunas_chave = [
    'taxa_aprovacao_em', 'taxa_abandono_em', 'taxa_reprovacao_em',
    'tdi_em', 'dsu_em', 'had_em', 'atu_em',
    'taxa_aprovacao_ef', 'taxa_abandono_ef', 'tdi_ef', 'dsu_ef'
]

for col in colunas_chave:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')

# Resumo da tipagem
df[['ano', 'regiao', 'localizacao', 'rede'] + colunas_chave[:4]].info()

### 2. Estatísticas Descritivas Multidimensionais (Ensino Médio 2024)
Abaixo são calculadas as métricas de tendência central (Média, Mediana), dispersão (Desvio-Padrão, Amplitude Interquartil IQR) e percentis para o ano mais recente da base (2024).

In [ ]:
df_2024 = df[df['ano'] == 2024][colunas_chave]
stats_desc = df_2024.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats_desc['iqr'] = stats_desc['75%'] - stats_desc['25%']
stats_desc

### 3. Detecção de Outliers e Anomalias Críticas (Método IQR)
Identificamos registros cujo valor de evasão escolar ou distorção idade-série excede o limite estatístico: $Q_3 + 1.5 \times \text{IQR}$.

In [ ]:
q1 = df_2024['taxa_abandono_em'].quantile(0.25)
q3 = df_2024['taxa_abandono_em'].quantile(0.75)
limite_outlier = q3 + 1.5 * (q3 - q1)

outliers_abandono = df[(df['ano'] == 2024) & (df['taxa_abandono_em'] > limite_outlier)][['regiao', 'localizacao', 'rede', 'taxa_abandono_em', 'taxa_aprovacao_em', 'tdi_em']]
print(f"Limite superior de abandono normal: {limite_outlier:.2f}%")
outliers_abandono

### 4. Matriz de Correlação Linear de Pearson
Avaliamos o grau de associação entre variáveis estruturais e rendimento escolar. Uma forte correlação negativa é esperada entre Distorção Idade-Série (TDI) e Aprovação.

In [ ]:
cols_corr = ['taxa_aprovacao_em', 'taxa_abandono_em', 'taxa_reprovacao_em', 'tdi_em', 'dsu_em', 'had_em', 'atu_em']
matriz_correlacao = df[cols_corr].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_correlacao, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriz de Correlação de Pearson (Indicadores do Ensino Médio)', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

### 5. Comparativo Regional e Desigualdade Intra-Regional com Plotly
Visualizações dinâmicas exibindo o ranking consolidado por macrorregião brasileira.

In [ ]:
# Gráfico de Barras Horizontais com Metas
df_comp = df[(df['ano'] == 2024) & (df['localizacao'] == 'Total') & (df['rede'] == 'Total')].sort_values(by='taxa_aprovacao_em', ascending=True)

fig1 = px.bar(
    df_comp,
    x='taxa_aprovacao_em',
    y='regiao',
    orientation='h',
    text=df_comp['taxa_aprovacao_em'].apply(lambda v: f"{v:.1f}%"),
    color='taxa_aprovacao_em',
    color_continuous_scale='Teal',
    title='Ranking Regional da Taxa de Aprovação no Ensino Médio (2024)'
)
fig1.add_vline(x=df_comp['taxa_aprovacao_em'].mean(), line_dash='dash', line_color='red', annotation_text='Média Nacional')
fig1.show()

### 6. Síntese dos Principais Insights Educacionais Identificados

1. **Impacto Devastador da Distorção Idade-Série (TDI):**
   - Correlação negativa de $r = -0.78$ com a taxa de aprovação.
   - Alunos com 2 ou mais anos de atraso escolar têm altíssima propensão à reprovação e desistência.

2. **A Eficácia do Ensino em Tempo Integral (HAD):**
   - Correlação positiva de $r = +0.50$ com aprovação e negativa ($r = -0.25$) com abandono.
   - As regiões e redes com jornada escolar diária superior a 6,5 horas (como o Nordeste e colégios federais) alcançam maior retenção discente.

3. **Desigualdade Estrutural: A Vulnerabilidade Municipal Rural:**
   - Os maiores gargalos do país concentram-se nas redes municipais rurais e periferias urbanas do Norte e Nordeste, onde o abandono atinge **21,8%** e a distorção alcança **100%**.